In [ ]:
import ee

PROJECT_ID = "lse"

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()  # opens an OAuth flow, click through and paste/allow
    ee.Initialize(project=PROJECT_ID)

print("EE initialized:", ee.String("ok").getInfo())

EE initialized: ok


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q segmentation-models-pytorch timm torchgeo mlflow rasterio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.4/811.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.2/891.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import hashlib
import glob
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import mlflow
import rasterio
from rasterio.windows import Window
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import glob
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchgeo.models import ViTSmall16_Weights
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import mlflow

In [ ]:
 #Study Area

# Small, representative agricultural sub-regions (~12-18 km across), NOT full states.
# Chosen for known cropping activity + reasonable cloud-free imagery availability.
#
# Punjab: Ludhiana district belt : wheat/paddy rotation (Rabi/Kharif)
# Maharashtra: Nashik belt : grape/onion/vegetable mixed agriculture
# Karnataka: Raichur belt : rice/cotton, Krishna-Tungabhadra command area


REGIONS = {
    "punjab_ludhiana": ee.Geometry.Rectangle([75.70, 30.80, 75.90, 30.98]),
    "maharashtra_nashik": ee.Geometry.Rectangle([73.70, 19.90, 73.90, 20.08]),
    "karnataka_raichur": ee.Geometry.Rectangle([76.30, 16.10, 76.50, 16.28]),
}

# Print area of each AOI as a sanity check
for name, geom in REGIONS.items():
    area_km2 = geom.area().divide(1e6).getInfo()
    print(f"{name}: {area_km2:.1f} km^2")

punjab_ludhiana: 382.0 km^2
maharashtra_nashik: 418.3 km^2
karnataka_raichur: 427.5 km^2


In [ ]:
# Seasonal date windows (captures phenology across the crop calendar)

# India's crop calendar roughly: Kharif (Jun-Oct), Rabi (Nov-Mar), Zaid/summer (Mar-Jun)
# 4 windows per region chosen to span sowing -> peak growth -> harvest -> fallow.

SEASONS = {
    "kharif_peak":        ("2023-09-01", "2023-09-30"),
    "kharif_harvest_rabi_sow": ("2023-11-15", "2023-12-15"),
    "rabi_peak":          ("2024-01-15", "2024-02-15"),
    "rabi_harvest_fallow": ("2024-04-01", "2024-04-30"),
}

In [ ]:
# Cloud masking (s2cloudless method)

CLOUD_FILTER = 60        # max scene-level cloud % to even consider
CLD_PRB_THRESH = 40      # cloud probability threshold (0-100)
NIR_DRK_THRESH = 0.15    # cloud shadow detection threshold on NIR
CLD_PRJ_DIST = 2         # km to project cloud shadows
BUFFER = 100             # m buffer around detected clouds

def get_s2_sr_cld_col(aoi, start_date, end_date):
    """Join Sentinel-2 SR (L2A) with the s2cloudless probability collection."""
    s2_sr_col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CLOUD_FILTER))
    )
    s2_cloudless_col = (
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
    )
    return ee.ImageCollection(
        ee.Join.saveFirst("s2cloudless").apply(
            primary=s2_sr_col,
            secondary=s2_cloudless_col,
            condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
        )
    )

def add_cloud_shadow_mask(img):
    cld_prb = ee.Image(img.get("s2cloudless")).select("probability")
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename("clouds")

    # Cloud shadow: dark NIR pixels projected from cloud edges
    dark_pixels = img.select("B8").lt(NIR_DRK_THRESH * 1e4).rename("dark_pixels")
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get("MEAN_SOLAR_AZIMUTH_ANGLE")))
    cld_proj = (
        is_cloud.directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST * 10)
        .reproject(crs=img.select("B2").projection(), scale=100)
        .select("distance")
        .mask()
        .rename("cloud_transform")
    )
    shadows = cld_proj.multiply(dark_pixels).rename("shadows")

    is_cld_shdw = is_cloud.add(shadows).gt(0)
    is_cld_shdw = (
        is_cld_shdw.focal_min(2).focal_max(BUFFER * 2 / 20)
        .reproject(crs=img.select("B2").projection(), scale=20)
        .rename("cloudmask")
    )
    return img.addBands(is_cld_shdw)

def apply_cloud_mask(img):
    not_cld_shdw = img.select("cloudmask").eq(0)
    return img.select(
        ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
    ).updateMask(not_cld_shdw)

def build_masked_composite(aoi, start_date, end_date):
    """Cloud-masked median composite for one region/season."""
    col = get_s2_sr_cld_col(aoi, start_date, end_date)
    masked = col.map(add_cloud_shadow_mask).map(apply_cloud_mask)
    return masked.median().clip(aoi)

In [ ]:
EXPORT_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
SCALE = 10  # meters/pixel (20m bands will be resampled to 10m on export)
BYTES_PER_PIXEL = 2  # Sentinel-2 SR bands are uint16

def dry_run():
    print(f"{'region':<22}{'season':<28}{'imgs':<6}{'est_MB':>10}")
    total_mb = 0
    plan = []
    for region_name, geom in REGIONS.items():
        area_m2 = geom.area().getInfo()
        n_pixels = area_m2 / (SCALE ** 2)
        est_bytes = n_pixels * len(EXPORT_BANDS) * BYTES_PER_PIXEL
        est_mb = est_bytes / 1e6

        for season_name, (start, end) in SEASONS.items():
            col = get_s2_sr_cld_col(geom, start, end)
            n_imgs = col.size().getInfo()
            print(f"{region_name:<22}{season_name:<28}{n_imgs:<6}{est_mb:>9.1f}")
            total_mb += est_mb
            plan.append(
                {
                    "region": region_name,
                    "season": season_name,
                    "start": start,
                    "end": end,
                    "n_source_images": n_imgs,
                    "est_mb": round(est_mb, 1),
                }
            )
    print(f"\nEstimated total export size: {total_mb:.1f} MB ({total_mb/1024:.2f} GB)")
    if total_mb / 1024 > 1.5:
        print("WARNING: exceeds 1.5 GB target — consider shrinking AOI or dropping a season.")
    return plan

plan = dry_run()


region                season                      imgs      est_MB
punjab_ludhiana       kharif_peak                 3          76.4
punjab_ludhiana       kharif_harvest_rabi_sow     6          76.4
punjab_ludhiana       rabi_peak                   4          76.4
punjab_ludhiana       rabi_harvest_fallow         5          76.4
maharashtra_nashik    kharif_peak                 3          83.7
maharashtra_nashik    kharif_harvest_rabi_sow     13         83.7
maharashtra_nashik    rabi_peak                   10         83.7
maharashtra_nashik    rabi_harvest_fallow         11         83.7
karnataka_raichur     kharif_peak                 3          85.5
karnataka_raichur     kharif_harvest_rabi_sow     9          85.5
karnataka_raichur     rabi_peak                   12         85.5
karnataka_raichur     rabi_harvest_fallow         8          85.5

Estimated total export size: 982.2 MB (0.96 GB)


In [ ]:
#EXPORT
RUN_EXPORT = True
DRIVE_FOLDER = "crop_classification_exports"

if RUN_EXPORT:
    tasks = []
    for region_name, geom in REGIONS.items():
        for season_name, (start, end) in SEASONS.items():
            composite = build_masked_composite(geom, start, end).select(EXPORT_BANDS)
            description = f"{region_name}_{season_name}"
            task = ee.batch.Export.image.toDrive(
                image=composite.toUint16(),
                description=description,
                folder=DRIVE_FOLDER,
                fileNamePrefix=description,
                region=geom,
                scale=SCALE,
                crs="EPSG:4326",
                maxPixels=1e9,
                fileFormat="GeoTIFF",
            )
            task.start()
            tasks.append(task)
            print(f"Started export task: {description}")

    print(f"\n{len(tasks)} export tasks submitted. Check progress at:")
    print("https://code.earthengine.google.com/tasks")
else:
    print("RUN_EXPORT is False — no exports triggered.")

Started export task: punjab_ludhiana_kharif_peak
Started export task: punjab_ludhiana_kharif_harvest_rabi_sow
Started export task: punjab_ludhiana_rabi_peak
Started export task: punjab_ludhiana_rabi_harvest_fallow
Started export task: maharashtra_nashik_kharif_peak
Started export task: maharashtra_nashik_kharif_harvest_rabi_sow
Started export task: maharashtra_nashik_rabi_peak
Started export task: maharashtra_nashik_rabi_harvest_fallow
Started export task: karnataka_raichur_kharif_peak
Started export task: karnataka_raichur_kharif_harvest_rabi_sow
Started export task: karnataka_raichur_rabi_peak
Started export task: karnataka_raichur_rabi_harvest_fallow

12 export tasks submitted. Check progress at:
https://code.earthengine.google.com/tasks


In [ ]:
# LABELS ESA WorldCover cropland mask, GEE-native:
#We are using the LULC for labelling as "cropland" or non-cropland"  where we only use a field called cropland class code
#we are considering this attribute as the ground truth
REGIONS = {
    "punjab_ludhiana": ee.Geometry.Rectangle([75.70, 30.80, 75.90, 30.98]),
    "maharashtra_nashik": ee.Geometry.Rectangle([73.70, 19.90, 73.90, 20.08]),
    "karnataka_raichur": ee.Geometry.Rectangle([76.30, 16.10, 76.50, 16.28]),
}

CROPLAND_CLASS_CODE = 40  # ESA WorldCover v200 "Cropland"
DRIVE_FOLDER = "crop_classification_exports"
RUN_EXPORT = True

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

def build_binary_mask(aoi):
    """1 = cropland, 0 = everything else, clipped to AOI."""
    return worldcover.eq(CROPLAND_CLASS_CODE).rename("cropland").clip(aoi).toUint8()

def dry_run():
    print(f"{'region':<22}{'cropland_%':>12}")
    for name, geom in REGIONS.items():
        mask = build_binary_mask(geom)
        stats = mask.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=10,
            maxPixels=1e9,
        ).getInfo()
        pct = stats.get("cropland", 0) * 100
        print(f"{name:<22}{pct:>11.1f}%")
        if pct < 5:
            print(f"  WARNING: {name} has very low cropland coverage — "
                  f"consider relocating the AOI.")

dry_run()



region                  cropland_%
punjab_ludhiana              52.1%
maharashtra_nashik           45.4%
karnataka_raichur            59.9%


In [ ]:
if RUN_EXPORT:
    tasks = []
    for name, geom in REGIONS.items():
        mask = build_binary_mask(geom)
        description = f"{name}_worldcover_mask"
        task = ee.batch.Export.image.toDrive(
            image=mask,
            description=description,
            folder=DRIVE_FOLDER,
            fileNamePrefix=description,
            region=geom,
            scale=10,
            crs="EPSG:4326",
            maxPixels=1e9,
            fileFormat="GeoTIFF",
        )
        task.start()
        tasks.append(task)
        print(f"Started export: {description}")
    print(f"\n{len(tasks)} mask export tasks submitted. Check "
          f"https://code.earthengine.google.com/tasks")
else:
    print("RUN_EXPORT is False : no exports triggered. "
          )

Started export: punjab_ludhiana_worldcover_mask
Started export: maharashtra_nashik_worldcover_mask
Started export: karnataka_raichur_worldcover_mask

3 mask export tasks submitted. Check https://code.earthengine.google.com/tasks


In [ ]:
# PREPROCESSING
DRIVE_DIR = "/content/drive/MyDrive/crop_classification_exports"
OUTPUT_DIR = "/content/patches"
PATCH_SIZE = 64
GRID_BLOCK_SIZE = 4  # patches per grid block edge, i.e. block = 4x4 patches
CROPLAND_FRACTION_THRESHOLD = 0.5  # patch labeled "cropland" if >=50% of pixels are cropland

REGIONS = ["punjab_ludhiana", "maharashtra_nashik", "karnataka_raichur"]
SEASONS = ["kharif_peak", "kharif_harvest_rabi_sow", "rabi_peak", "rabi_harvest_fallow"]

S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
# Band index positions within the exported GeoTIFF
BLUE, GREEN, RED, RE1, RE2, RE3, NIR, RE4, SWIR1, SWIR2 = range(10)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Spectral indices
def compute_indices(bands, eps=1e-6):
    """
    Spectral Indices are derieved from Sentinel -2,
    Nir = Band (B8)
    Red band= Band (B4)
    Green band= Band (B3)
    Blue band= Band (B2)

    bands: float32 array [C, H, W] in Sentinel-2 SR reflectance*10000 units,
           channel order matching S2_BANDS above.
    Returns NDVI, NDWI, EVI stacked as [3, H, W], each roughly in [-1, 1]
    (EVI is not strictly bounded but typically falls in that range for
    vegetated land).
    """
    b = bands.astype(np.float32) / 10000.0  # scale to reflectance [0, ~1]
    red, nir, green, blue = b[RED], b[NIR], b[GREEN], b[BLUE]

    ndvi = (nir - red) / (nir + red + eps)
    ndwi = (green - nir) / (green + nir + eps)  # McFeeters NDWI (water/moisture)
    evi = 2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1 + eps)

    return np.stack([ndvi, ndwi, evi], axis=0)

In [ ]:
# Patch extraction

def block_id_for_pixel(row, col, patch_size=PATCH_SIZE, block_size=GRID_BLOCK_SIZE):
    patch_row, patch_col = row // patch_size, col // patch_size
    return (patch_row // block_size, patch_col // block_size)


def assign_split(region, block_row, block_col, train_frac=0.70, val_frac=0.15):
    """Deterministic hash-based split assignment per (region, block) —
    reproducible across runs without storing a lookup table."""
    key = f"{region}_{block_row}_{block_col}".encode()
    h = int(hashlib.md5(key).hexdigest(), 16) % 1000 / 1000.0
    if h < train_frac:
        return "train"
    elif h < train_frac + val_frac:
        return "val"
    else:
        return "test"


def extract_patches_for_region_season(region, season):
    img_path = os.path.join(DRIVE_DIR, f"{region}_{season}.tif")
    mask_path = os.path.join(DRIVE_DIR, f"{region}_worldcover_mask.tif")

    if not os.path.exists(img_path):
        print(f"  SKIP (missing): {img_path}")
        return []
    if not os.path.exists(mask_path):
        print(f"  SKIP (missing mask): {mask_path}")
        return []

    records = []
    with rasterio.open(img_path) as img_src, rasterio.open(mask_path) as mask_src:
        h, w = img_src.height, img_src.width
        n_rows, n_cols = h // PATCH_SIZE, w // PATCH_SIZE

        for pr in range(n_rows):
            for pc in range(n_cols):
                row_off, col_off = pr * PATCH_SIZE, pc * PATCH_SIZE
                window = Window(col_off, row_off, PATCH_SIZE, PATCH_SIZE)

                img_patch = img_src.read(window=window).astype(np.float32)  # [10, P, P]
                if img_patch.shape[1:] != (PATCH_SIZE, PATCH_SIZE):
                    continue  # partial edge patch, skip
                if np.all(img_patch == 0):
                    continue  # fully masked/no-data patch (cloud-masked composite gap)

                # Mask is exported at the same 10m/EPSG:4326 grid as the imagery,
                # so pixel windows align directly.
                mask_patch = mask_src.read(1, window=window)
                if mask_patch.shape != (PATCH_SIZE, PATCH_SIZE):
                    continue
                cropland_frac = float(mask_patch.mean())
                label = int(cropland_frac >= CROPLAND_FRACTION_THRESHOLD)

                indices = compute_indices(img_patch)
                stacked = np.concatenate([img_patch, indices], axis=0)  # [13, P, P]

                block_row, block_col = block_id_for_pixel(row_off, col_off)
                split = assign_split(region, block_row, block_col)

                out_name = f"{region}_{season}_r{pr}_c{pc}.npz"
                out_path = os.path.join(OUTPUT_DIR, split, out_name)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                np.savez_compressed(
                    out_path,
                    features=stacked.astype(np.float32),
                    pixel_mask=mask_patch.astype(np.uint8),
                    label=label,
                    cropland_frac=cropland_frac,
                    region=region,
                    season=season,
                )
                records.append({"path": out_path, "split": split, "label": label,
                                 "region": region, "season": season})
    return records


def run_preprocessing():
    all_records = []
    for region in REGIONS:
        for season in SEASONS:
            print(f"Processing {region} / {season} ...")
            recs = extract_patches_for_region_season(region, season)
            print(f"  -> {len(recs)} patches")
            all_records.extend(recs)

    counts = {}
    for r in all_records:
        key = (r["split"], r["label"])
        counts[key] = counts.get(key, 0) + 1

    print("\n=== Patch summary ===")
    print(f"Total patches: {len(all_records)}")
    for split in ["train", "val", "test"]:
        n_crop = counts.get((split, 1), 0)
        n_noncrop = counts.get((split, 0), 0)
        total = n_crop + n_noncrop
        print(f"  {split:<6}: {total:>5} patches  "
              f"(cropland={n_crop}, non-cropland={n_noncrop})")
        if total == 0:
            print(f"    WARNING: no patches in '{split}' split — "
                  f"check that Stage 1/2 exports downloaded correctly.")
    return all_records


if __name__ == "__main__":
    records = run_preprocessing()


Processing punjab_ludhiana / kharif_peak ...
  -> 1054 patches
Processing punjab_ludhiana / kharif_harvest_rabi_sow ...
  -> 1053 patches
Processing punjab_ludhiana / rabi_peak ...
  -> 1054 patches
Processing punjab_ludhiana / rabi_harvest_fallow ...
  -> 1054 patches
Processing maharashtra_nashik / kharif_peak ...
  -> 714 patches
Processing maharashtra_nashik / kharif_harvest_rabi_sow ...
  -> 1054 patches
Processing maharashtra_nashik / rabi_peak ...
  -> 1054 patches
Processing maharashtra_nashik / rabi_harvest_fallow ...
  -> 1054 patches
Processing karnataka_raichur / kharif_peak ...
  -> 992 patches
Processing karnataka_raichur / kharif_harvest_rabi_sow ...
  -> 1054 patches
Processing karnataka_raichur / rabi_peak ...
  -> 1054 patches
Processing karnataka_raichur / rabi_harvest_fallow ...
  -> 1054 patches

=== Patch summary ===
Total patches: 12245
  train :  7740 patches  (cropland=4568, non-cropland=3172)
  val   :  2099 patches  (cropland=1182, non-cropland=917)
  test  :

In [ ]:
#Model 1 — RANDOM FOREST BASELINE
PATCH_DIR = "/content/patches"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]


def load_split(split):
    paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
    X, y = [], []
    for p in paths:
        d = np.load(p)
        feats = d["features"]  # [13, H, W]
        means = feats.mean(axis=(1, 2))
        stds = feats.std(axis=(1, 2))
        X.append(np.concatenate([means, stds]))
        y.append(int(d["label"]))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), paths


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    print("Loading patches...")
    X_train, y_train, _ = load_split("train")
    X_val, y_val, _ = load_split("val")
    X_test, y_test, _ = load_split("test")
    print(f"train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")

    if len(y_train) == 0 or len(y_test) == 0:
        raise RuntimeError(
            "No train/test patches found "
            f"and confirm .npz files exist under {PATCH_DIR}."
        )

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="random_forest_baseline"):
        params = dict(n_estimators=300, max_depth=None, min_samples_leaf=2,
                       class_weight="balanced", random_state=42, n_jobs=-1)
        mlflow.log_params(params)

        clf = RandomForestClassifier(**params)
        clf.fit(X_train, y_train)

        val_pred = clf.predict(X_val) if len(y_val) else None
        test_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, test_pred)
        f1_macro = f1_score(y_test, test_pred, average="macro")
        f1_per_class = f1_score(y_test, test_pred, average=None)
        cm = confusion_matrix(y_test, test_pred)

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric(f"test_f1_{cname}", f1c)
        if val_pred is not None and len(y_val):
            mlflow.log_metric("val_accuracy", accuracy_score(y_val, val_pred))

        print("\n=== Random Forest — Test Set ===")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1_macro:.4f}")
        print(classification_report(y_test, test_pred, target_names=CLASS_NAMES))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "random_forest",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(y_train), "n_val": len(y_val), "n_test": len(y_test),
        })

    return clf


if __name__ == "__main__":
    main()


Loading patches...
train=7740  val=2099  test=2406


2026/09/07 08:41:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/07 08:41:15 INFO mlflow.store.db.utils: Updating database tables
2026/09/07 08:41:17 INFO mlflow.tracking.fluent: Experiment with name 'crop_classification' does not exist. Creating a new experiment.



=== Random Forest — Test Set ===
Accuracy: 0.8662
Macro F1: 0.8661
              precision    recall  f1-score   support

non_cropland       0.87      0.85      0.86      1179
    cropland       0.86      0.88      0.87      1227

    accuracy                           0.87      2406
   macro avg       0.87      0.87      0.87      2406
weighted avg       0.87      0.87      0.87      2406

Confusion matrix (rows=true, cols=pred):
[[1008  171]
 [ 151 1076]]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Model 2 — UNET BASELINE
PATCH_DIR = "/content/patches"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3
IN_CHANNELS = 13  # 10 S2 bands + NDVI + NDWI + EVI

print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU detected: running on CPU. ")


class PatchDataset(Dataset):
    def __init__(self, split):
        self.paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
        if len(self.paths) == 0:
            raise RuntimeError(f"No patches found for split='{split}' in {PATCH_DIR}. "
                                f"Run preprocessing.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        feats = d["features"].astype(np.float32)  # [13, H, W]
        # Per-sample normalization (mean/std) — simple and robust without
        # needing a separately-computed training-set-wide statistic.
        mean = feats.mean(axis=(1, 2), keepdims=True)
        std = feats.std(axis=(1, 2), keepdims=True) + 1e-6
        feats = (feats - mean) / std

        pixel_mask = d["pixel_mask"].astype(np.float32)  # [H, W], 0/1
        patch_label = int(d["label"])
        return torch.from_numpy(feats), torch.from_numpy(pixel_mask), patch_label


def build_model():
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,  # ImageNet weights don't apply to 13-channel input
        in_channels=IN_CHANNELS,
        classes=1,  # binary segmentation, single-channel logit
    )
    return model.to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for feats, pixel_mask, _ in loader:
        feats, pixel_mask = feats.to(DEVICE), pixel_mask.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feats).squeeze(1)  # [B, H, W]
        loss = criterion(logits, pixel_mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * feats.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_patch_preds, all_patch_labels = [], []
    for feats, pixel_mask, patch_label in loader:
        feats = feats.to(DEVICE)
        logits = model(feats).squeeze(1)
        pixel_probs = torch.sigmoid(logits)
        pixel_preds = (pixel_probs > 0.5).float().cpu()
        # Aggregate to patch level: majority of predicted-cropland pixels
        patch_pred = (pixel_preds.mean(dim=(1, 2)) > 0.5).long().numpy()
        all_patch_preds.extend(patch_pred.tolist())
        all_patch_labels.extend(patch_label.numpy().tolist())
    return np.array(all_patch_labels), np.array(all_patch_preds)


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    train_ds = PatchDataset("train")
    val_ds = PatchDataset("val")
    test_ds = PatchDataset("test")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="unet_segmentation"):
        mlflow.log_params(dict(
            encoder="resnet18", in_channels=IN_CHANNELS, epochs=EPOCHS,
            batch_size=BATCH_SIZE, lr=LR, device=str(DEVICE),
        ))

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            val_labels, val_preds = evaluate(model, val_loader)
            val_acc = accuracy_score(val_labels, val_preds) if len(val_labels) else float("nan")
            print(f"Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        test_labels, test_preds = evaluate(model, test_loader)
        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_per_class = f1_score(test_labels, test_preds, average=None, labels=[0, 1])
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric(f"test_f1_{cname}", f1c)

        print("\n=== UNet  bASE===")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1_macro:.4f}")
        print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, labels=[0, 1]))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "unet",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        })

    return model


if __name__ == "__main__":
    main()

Using device: cuda
Epoch 1/15  train_loss=0.5069  val_acc=0.7975
Epoch 2/15  train_loss=0.4561  val_acc=0.8414
Epoch 3/15  train_loss=0.4366  val_acc=0.8761
Epoch 4/15  train_loss=0.4279  val_acc=0.8757
Epoch 5/15  train_loss=0.4147  val_acc=0.8509
Epoch 6/15  train_loss=0.4034  val_acc=0.8518
Epoch 7/15  train_loss=0.3999  val_acc=0.8604
Epoch 8/15  train_loss=0.3918  val_acc=0.8823
Epoch 9/15  train_loss=0.3852  val_acc=0.8471
Epoch 10/15  train_loss=0.3807  val_acc=0.8414
Epoch 11/15  train_loss=0.3668  val_acc=0.8737
Epoch 12/15  train_loss=0.3561  val_acc=0.8771
Epoch 13/15  train_loss=0.3534  val_acc=0.8080
Epoch 14/15  train_loss=0.3565  val_acc=0.8699
Epoch 15/15  train_loss=0.3363  val_acc=0.8790

=== UNet — Test Set (patch-level, aggregated from pixel predictions) ===
Accuracy: 0.8853
Macro F1: 0.8846
              precision    recall  f1-score   support

non_cropland       0.93      0.82      0.88      1179
    cropland       0.85      0.94      0.89      1227

    accuracy 

In [ ]:
#MODEL 3: GEOSPATIAL FOUNDATION MODEL, Pretrained VIT SSL4EO-S12 MAE

PATCH_DIR = "/content/patches"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4  # smaller LR — fine-tuning a pretrained encoder, not training from scratch
FREEZE_ENCODER_EPOCHS = 2  # linear-probe warmup before unfreezing, for stability

print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU detected, Using CPU)

# Our exported band order (Stage 1 EXPORT_BANDS) -> position in the
# 13-band order the pretrained model expects.
OUR_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
TARGET_BANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8a", "B9", "B10", "B11", "B12"]
# map our band name -> target band name (B8A vs B8a casing)
NAME_FIX = {"B8A": "B8a"}
OUR_TO_TARGET_IDX = [TARGET_BANDS.index(NAME_FIX.get(b, b)) for b in OUR_BANDS]


def remap_to_13_band(patch_10band):
    """patch_10band: [10, H, W] -> zero-padded [13, H, W] in TARGET_BANDS order."""
    c, h, w = patch_10band.shape
    out = np.zeros((13, h, w), dtype=np.float32)
    for src_idx, tgt_idx in enumerate(OUR_TO_TARGET_IDX):
        out[tgt_idx] = patch_10band[src_idx]
    return out


class PatchDataset(Dataset):
    def __init__(self, split):
        self.paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
        if len(self.paths) == 0:
            raise RuntimeError(f"No patches found for split='{split}' in {PATCH_DIR}. "
                                f"Go to preprocessing")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        feats = d["features"].astype(np.float32)[:10]  # drop the 3 index channels, raw bands only
        feats = remap_to_13_band(feats)
        mean = feats.mean(axis=(1, 2), keepdims=True)
        std = feats.std(axis=(1, 2), keepdims=True) + 1e-6
        feats = (feats - mean) / std
        label = int(d["label"])
        return torch.from_numpy(feats), label


def build_model():
    weights = ViTSmall16_Weights.SENTINEL2_ALL_MAE
    model = timm.create_model("vit_small_patch16_224", in_chans=13, num_classes=2)
    try:
        state_dict = weights.get_state_dict(progress=True)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f"Loaded pretrained SSL4EO-S12 MAE weights. "
              f"({len(missing)} params randomly init'd, e.g. classifier head)")
    except Exception as e:
        print(f"WARNING: could not download pretrained weights ({e}). ")
    return model.to(DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for feats, labels in loader:
        feats = F.interpolate(feats, size=(224, 224), mode="bilinear", align_corners=False)
        feats, labels = feats.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * feats.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for feats, labels in loader:
        feats = F.interpolate(feats, size=(224, 224), mode="bilinear", align_corners=False)
        feats = feats.to(DEVICE)
        logits = model(feats)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_preds)


def set_encoder_trainable(model, trainable):
    for name, param in model.named_parameters():
        if "head" not in name:  # timm ViT classifier head is named "head"
            param.requires_grad = trainable


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    train_ds = PatchDataset("train")
    val_ds = PatchDataset("val")
    test_ds = PatchDataset("test")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="foundation_model_vit_ssl4eo_s12_mae"):
        mlflow.log_params(dict(
            base_model="ViTSmall16_Weights.SENTINEL2_ALL_MAE (SatMAE substitute)",
            epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=str(DEVICE),
            freeze_encoder_epochs=FREEZE_ENCODER_EPOCHS,
        ))

        for epoch in range(1, EPOCHS + 1):
            set_encoder_trainable(model, trainable=(epoch > FREEZE_ENCODER_EPOCHS))
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            val_labels, val_preds = evaluate(model, val_loader)
            val_acc = accuracy_score(val_labels, val_preds) if len(val_labels) else float("nan")
            phase = "linear-probe" if epoch <= FREEZE_ENCODER_EPOCHS else "fine-tune"
            print(f"Epoch {epoch}/{EPOCHS} [{phase}]  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        test_labels, test_preds = evaluate(model, test_loader)
        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_per_class = f1_score(test_labels, test_preds, average=None, labels=[0, 1])
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric(f"test_f1_{cname}", f1c)

        print("\n=== Foundation Model (ViT / SSL4EO-S12 MAE)  ===")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1_macro:.4f}")
        print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, labels=[0, 1]))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "foundation_model_vit_ssl4eo_s12_mae",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        })

    return model


if __name__ == "__main__":
    main()

Using device: cuda
Downloading: "https://huggingface.co/wangyi111/SSL4EO-S12/resolve/75c72195d35201dc1fb210818993518c25da566b/B13_vits16_mae_ep99_enc.pth" to /root/.cache/torch/hub/checkpoints/B13_vits16_mae_ep99_enc.pth


100%|██████████| 86.5M/86.5M [00:01<00:00, 56.7MB/s]


Loaded pretrained SSL4EO-S12 MAE weights. (2 params randomly init'd, e.g. classifier head)
Epoch 1/10 [linear-probe]  train_loss=0.5807  val_acc=0.7189
Epoch 2/10 [linear-probe]  train_loss=0.5171  val_acc=0.7489
Epoch 3/10 [fine-tune]  train_loss=0.4268  val_acc=0.7880
Epoch 4/10 [fine-tune]  train_loss=0.3300  val_acc=0.7794
Epoch 5/10 [fine-tune]  train_loss=0.2865  val_acc=0.8437
Epoch 6/10 [fine-tune]  train_loss=0.2452  val_acc=0.8761
Epoch 7/10 [fine-tune]  train_loss=0.2167  val_acc=0.8442
Epoch 8/10 [fine-tune]  train_loss=0.1902  val_acc=0.8780
Epoch 9/10 [fine-tune]  train_loss=0.1650  val_acc=0.8595
Epoch 10/10 [fine-tune]  train_loss=0.1534  val_acc=0.8609

=== Foundation Model (ViT / SSL4EO-S12 MAE) — Test Set ===
Accuracy: 0.8799
Macro F1: 0.8797
              precision    recall  f1-score   support

non_cropland       0.90      0.85      0.87      1179
    cropland       0.87      0.90      0.88      1227

    accuracy                           0.88      2406
   macro a

In [ ]:
# RESULTS COMPARISON OF 3 MODELS (RF, UNET, VIT)
RESULTS_PATH = "/content/results.json"
OUTPUT_DIR = "/content/deliverables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_DISPLAY_NAMES = {
    "random_forest": "Random Forest\n(hand-crafted features)",
    "unet": "UNet\n(ResNet18 encoder)",
    "foundation_model_vit_ssl4eo_s12_mae": "ViT Foundation Model\n(SSL4EO-S12 MAE, SatMAE substitute)",
}


def load_results():
    if not os.path.exists(RESULTS_PATH):
        raise RuntimeError(
            f"{RESULTS_PATH} not found "
            f"appends its results to this file."
        )
    with open(RESULTS_PATH) as f:
        results = json.load(f)

    # keep only the LATEST run per model (in case a stage was re-run)
    latest = {}
    for r in results:
        latest[r["model"]] = r
    return list(latest.values())


def build_table(results):
    rows = []
    for r in results:
        row = {
            "Model": MODEL_DISPLAY_NAMES.get(r["model"], r["model"]).replace("\n", " "),
            "Accuracy": round(r["test_accuracy"], 4),
            "Macro F1": round(r["test_f1_macro"], 4),
            "F1 (non-cropland)": round(r["test_f1_per_class"].get("non_cropland", float("nan")), 4),
            "F1 (cropland)": round(r["test_f1_per_class"].get("cropland", float("nan")), 4),
            "N test patches": r.get("n_test", "-"),
        }
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
    return df


def plot_comparison(results, out_path):
    models = [MODEL_DISPLAY_NAMES.get(r["model"], r["model"]) for r in results]
    acc = [r["test_accuracy"] for r in results]
    f1 = [r["test_f1_macro"] for r in results]

    x = range(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([i - width / 2 for i in x], acc, width, label="Accuracy")
    ax.bar([i + width / 2 for i in x], f1, width, label="Macro F1")
    ax.set_xticks(list(x))
    ax.set_xticklabels(models, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Crop (Cropland) Classification — Model Comparison, Test Set")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def main():
    results = load_results()
    present = {r["model"] for r in results}
    expected = set(MODEL_DISPLAY_NAMES.keys())
    missing = expected - present
    if missing:
        print(f"WARNING: missing results for: {sorted(missing)} — "
              f"run those Stage 4 scripts before treating this as final.")

    df = build_table(results)
    print("\n=== Model Comparison ===")
    print(df.to_string(index=False))

    csv_path = os.path.join(OUTPUT_DIR, "results_comparison_table.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nSaved table: {csv_path}")

    chart_path = os.path.join(OUTPUT_DIR, "results_comparison_chart.png")
    plot_comparison(results, chart_path)
    print(f"Saved chart: {chart_path}")

    return df


if __name__ == "__main__":
    main()


=== Model Comparison ===
                                                   Model  Accuracy  Macro F1  F1 (non-cropland)  F1 (cropland)  N test patches
                                 UNet (ResNet18 encoder)    0.8853    0.8846             0.8757         0.8935            2406
ViT Foundation Model (SSL4EO-S12 MAE, SatMAE substitute)    0.8799    0.8797             0.8746         0.8847            2406
                   Random Forest (hand-crafted features)    0.8662    0.8661             0.8623         0.8698            2406

Saved table: /content/deliverables/results_comparison_table.csv
Saved chart: /content/deliverables/results_comparison_chart.png


In [ ]:
# SENTINEL-1 SAR INTEGRATION


# To sidestep clash of `REGIONS` and `SEASONS`, everything below uses its
# OWN names instead of trusting the ambient REGIONS / SEASONS globals.

S1_REGION_GEOMS = {
    "punjab_ludhiana": ee.Geometry.Rectangle([75.70, 30.80, 75.90, 30.98]),
    "maharashtra_nashik": ee.Geometry.Rectangle([73.70, 19.90, 73.90, 20.08]),
    "karnataka_raichur": ee.Geometry.Rectangle([76.30, 16.10, 76.50, 16.28]),
}

S1_SEASON_WINDOWS = {
    "kharif_peak":             ("2023-09-01", "2023-09-30"),
    "kharif_harvest_rabi_sow": ("2023-11-15", "2023-12-15"),
    "rabi_peak":               ("2024-01-15", "2024-02-15"),
    "rabi_harvest_fallow":     ("2024-04-01", "2024-04-30"),
}

# Region/season name lists
OPT_SAR_REGIONS = ["punjab_ludhiana", "maharashtra_nashik", "karnataka_raichur"]
OPT_SAR_SEASONS = ["kharif_peak", "kharif_harvest_rabi_sow", "rabi_peak", "rabi_harvest_fallow"]

S1_SCALE = 10          # MUST match S1's SCALE, or pixel grids won't line up
S1_CRS = "EPSG:4326"   # MUST match S1's export CRS
DRIVE_FOLDER = "crop_classification_exports"  #

SPECKLE_FILTER = "focal_median"
FOCAL_MEDIAN_RADIUS = 50          # meters (~5x5 px at 10m)


In [ ]:
# Speckle filter + Sentinel-1 composite builder


def to_natural(img):
    return ee.Image(10.0).pow(img.divide(10.0))

def to_db(img):
    return img.log10().multiply(10.0)

def build_s1_composite(aoi, start_date, end_date):
    col = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
    )

    def filter_speckle(img):
        natural = to_natural(img)
        if SPECKLE_FILTER == "focal_median":
            filtered = natural.focal_median(radius=FOCAL_MEDIAN_RADIUS, units="meters")
        else:
            raise ValueError(
                "SPECKLE_FILTER=%r isn't implemented here."
                "Use 'focal_median'." % SPECKLE_FILTER
            )
        return to_db(filtered).copyProperties(img, img.propertyNames())

    filtered_col = col.map(filter_speckle)
    # Median composite across the season window -- same temporal
    # aggregation strategy we already used for the Sentinel-2 composites.
    return filtered_col.median().clip(aoi).rename(["VV", "VH"])


def dry_run_s1():
    header = "{:<22}{:<28}{:<8}".format("region", "season", "n_imgs")
    print(header)
    plan = []
    for region_name, geom in S1_REGION_GEOMS.items():
        for season_name, (start, end) in S1_SEASON_WINDOWS.items():
            col = (
                ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterBounds(geom).filterDate(start, end)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
            )
            n = col.size().getInfo()
            print("{:<22}{:<28}{:<8}".format(region_name, season_name, n))
            plan.append({"region": region_name, "season": season_name, "n_source_images": n})
            if n == 0:
                print("  WARNING: zero Sentinel-1 scenes for " + region_name + "/" + season_name +
                      " check orbit coverage / date window. Export will yield an "
                      "empty/no-data image and Stage 5b will skip this region-season.")
    return plan

s1_plan = dry_run_s1()


region                season                      n_imgs  
punjab_ludhiana       kharif_peak                 9       
punjab_ludhiana       kharif_harvest_rabi_sow     8       
punjab_ludhiana       rabi_peak                   8       
punjab_ludhiana       rabi_harvest_fallow         10      
maharashtra_nashik    kharif_peak                 4       
maharashtra_nashik    kharif_harvest_rabi_sow     3       
maharashtra_nashik    rabi_peak                   3       
maharashtra_nashik    rabi_harvest_fallow         5       
karnataka_raichur     kharif_peak                 2       
karnataka_raichur     kharif_harvest_rabi_sow     1       
karnataka_raichur     rabi_peak                   2       
karnataka_raichur     rabi_harvest_fallow         2       


In [ ]:
# Trigger the Sentinel-1 exports


RUN_S1_EXPORT = True

if RUN_S1_EXPORT:
    s1_tasks = []
    for region_name, geom in S1_REGION_GEOMS.items():
        for season_name, (start, end) in S1_SEASON_WINDOWS.items():
            composite = build_s1_composite(geom, start, end)
            description = region_name + "_" + season_name + "_s1"
            task = ee.batch.Export.image.toDrive(
                image=composite.toFloat(),
                description=description,
                folder=DRIVE_FOLDER,
                fileNamePrefix=description,
                region=geom,          # SAME geometry object as the S2/mask exports
                scale=S1_SCALE,       # SAME scale
                crs=S1_CRS,           # SAME CRS
                maxPixels=1e9,
            )
            task.start()
            s1_tasks.append(task)
            print("Started export: " + description)
    print("\n" + str(len(s1_tasks)) + " Sentinel-1 export tasks submitted. Check")
    print("https://code.earthengine.google.com/tasks")
else:
    print("RUN_S1_EXPORT is False.")


Started export: punjab_ludhiana_kharif_peak_s1
Started export: punjab_ludhiana_kharif_harvest_rabi_sow_s1
Started export: punjab_ludhiana_rabi_peak_s1
Started export: punjab_ludhiana_rabi_harvest_fallow_s1
Started export: maharashtra_nashik_kharif_peak_s1
Started export: maharashtra_nashik_kharif_harvest_rabi_sow_s1
Started export: maharashtra_nashik_rabi_peak_s1
Started export: maharashtra_nashik_rabi_harvest_fallow_s1
Started export: karnataka_raichur_kharif_peak_s1
Started export: karnataka_raichur_kharif_harvest_rabi_sow_s1
Started export: karnataka_raichur_rabi_peak_s1
Started export: karnataka_raichur_rabi_harvest_fallow_s1

12 Sentinel-1 export tasks submitted. Check
https://code.earthengine.google.com/tasks


In [ ]:
# Align Sentinel-1 to the EXISTING patch grid / block split

# Reuses block_id_for_pixel() and assign_split()  UNCHANGED
# same region + same block row/col -> same split, so optical-only

# Also reuses compute_indices() unchanged (NDVI/NDWI/EVI
# computed from the 10 raw S2 bands, exactly as before) and DRIVE_DIR /
# PATCH_SIZE / CROPLAND_FRACTION_THRESHOLD from that same stage.

OUT_DIR_OPT_SAR = "/content/patches_opt_sar"
os.makedirs(OUT_DIR_OPT_SAR, exist_ok=True)


def extract_optical_sar_patches_for_region_season(region, season):
    img_path = os.path.join(DRIVE_DIR, region + "_" + season + ".tif")
    mask_path = os.path.join(DRIVE_DIR, region + "_worldcover_mask.tif")
    s1_path = os.path.join(DRIVE_DIR, region + "_" + season + "_s1.tif")

    for p in (img_path, mask_path, s1_path):
        if not os.path.exists(p):
            print("  SKIP (missing): " + p)
            return []

    records = []
    with rasterio.open(img_path) as img_src, \
         rasterio.open(mask_path) as mask_src, \
         rasterio.open(s1_path) as s1_src:

        if (img_src.width, img_src.height) != (s1_src.width, s1_src.height):
            print("  SKIP (grid mismatch) " + region + "/" + season +
                  ": S2=" + str(img_src.width) + "x" + str(img_src.height) +
                  " vs S1=" + str(s1_src.width) + "x" + str(s1_src.height) +
                  " re-export S1 with the exact same region/scale/crs as "
                  "the S2 composite.")
            return []

        h, w = img_src.height, img_src.width
        n_rows, n_cols = h // PATCH_SIZE, w // PATCH_SIZE

        for pr in range(n_rows):
            for pc in range(n_cols):
                row_off, col_off = pr * PATCH_SIZE, pc * PATCH_SIZE
                window = Window(col_off, row_off, PATCH_SIZE, PATCH_SIZE)

                img_patch = img_src.read(window=window).astype(np.float32)
                if img_patch.shape[1:] != (PATCH_SIZE, PATCH_SIZE):
                    continue
                if np.all(img_patch == 0):
                    continue

                s1_patch = s1_src.read(window=window).astype(np.float32)  # [2, P, P] VV, VH in dB
                if s1_patch.shape[1:] != (PATCH_SIZE, PATCH_SIZE):
                    continue
                if not np.isfinite(s1_patch).all():
                    # focal_median can leave no-data slivers at AOI edges
                    s1_patch = np.nan_to_num(s1_patch, nan=0.0, posinf=0.0, neginf=0.0)

                mask_patch = mask_src.read(1, window=window)
                if mask_patch.shape != (PATCH_SIZE, PATCH_SIZE):
                    continue
                cropland_frac = float(mask_patch.mean())
                label = int(cropland_frac >= CROPLAND_FRACTION_THRESHOLD)

                indices = compute_indices(img_patch)
                stacked = np.concatenate([img_patch, indices, s1_patch], axis=0)  # [15, P, P]

                block_row, block_col = block_id_for_pixel(row_off, col_off)  # reused
                split = assign_split(region, block_row, block_col)          # reused

                out_name = region + "_" + season + "_r" + str(pr) + "_c" + str(pc) + ".npz"
                out_path = os.path.join(OUT_DIR_OPT_SAR, split, out_name)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                np.savez_compressed(
                    out_path,
                    features=stacked.astype(np.float32),
                    pixel_mask=mask_patch.astype(np.uint8),
                    label=label,
                    cropland_frac=cropland_frac,
                    region=region,
                    season=season,
                )
                records.append({"path": out_path, "split": split, "label": label,
                                 "region": region, "season": season})
    return records


def run_preprocessing_opt_sar():
    all_records = []
    for region in OPT_SAR_REGIONS:
        for season in OPT_SAR_SEASONS:
            print("Processing " + region + " / " + season + " ...")
            recs = extract_optical_sar_patches_for_region_season(region, season)
            print("  -> " + str(len(recs)) + " patches")
            all_records.extend(recs)

    counts = {}
    for r in all_records:
        key = (r["split"], r["label"])
        counts[key] = counts.get(key, 0) + 1
    print("\n=== Optical+SAR patch summary ===")
    print("Total patches: " + str(len(all_records)))
    for split in ["train", "val", "test"]:
        n_crop = counts.get((split, 1), 0)
        n_noncrop = counts.get((split, 0), 0)
        total = n_crop + n_noncrop
        print("  {:<6}: {:>5} patches (cropland={}, non-cropland={})".format(
            split, total, n_crop, n_noncrop))
    print("\nCompare these counts to original optical")
    print("they should be close but not necessarily identical (S1 AOI-edge")

    return all_records


if __name__ == "__main__":
    opt_sar_records = run_preprocessing_opt_sar()


Processing punjab_ludhiana / kharif_peak ...
  -> 1054 patches
Processing punjab_ludhiana / kharif_harvest_rabi_sow ...
  -> 1053 patches
Processing punjab_ludhiana / rabi_peak ...
  -> 1054 patches
Processing punjab_ludhiana / rabi_harvest_fallow ...
  -> 1054 patches
Processing maharashtra_nashik / kharif_peak ...
  -> 714 patches
Processing maharashtra_nashik / kharif_harvest_rabi_sow ...
  -> 1054 patches
Processing maharashtra_nashik / rabi_peak ...
  -> 1054 patches
Processing maharashtra_nashik / rabi_harvest_fallow ...
  -> 1054 patches
Processing karnataka_raichur / kharif_peak ...
  -> 992 patches
Processing karnataka_raichur / kharif_harvest_rabi_sow ...
  -> 1054 patches
Processing karnataka_raichur / rabi_peak ...
  -> 1054 patches
Processing karnataka_raichur / rabi_harvest_fallow ...
  -> 1054 patches

=== Optical+SAR patch summary ===
Total patches: 12245
  train :  7740 patches (cropland=4568, non-cropland=3172)
  val   :  2099 patches (cropland=1182, non-cropland=917)

In [ ]:
# Retrain UNet on optical+SAR (15 channels) vs optical-only
#
# Architecture, loss, optimizer, and epoch count are IDENTICAL to
# existing UNet run (encoder="resnet18", 15 epochs, Adam,
# lr=1e-3, BCEWithLogitsLoss, pixel->patch aggregation) the only
# difference is IN_CHANNELS (15 vs 13) and the data directory.


PATCH_DIR_OPT_SAR = "/content/patches_opt_sar"
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OPT_SAR_BATCH_SIZE = 32
OPT_SAR_EPOCHS = 15
OPT_SAR_LR = 1e-3
OPT_SAR_IN_CHANNELS = 15  # 10 S2 bands + NDVI/NDWI/EVI + VV_dB + VH_dB

print("Using device:", DEVICE)
if DEVICE.type == "cpu":
    print("No GPU detected -- training will run on CPU and be slower. "
          "In Colab: Runtime > Change runtime type > GPU (T4), then re-run this cell.")


class PatchDatasetOptSar(Dataset):
    def __init__(self, split):
        self.paths = sorted(glob.glob(os.path.join(PATCH_DIR_OPT_SAR, split, "*.npz")))
        if len(self.paths) == 0:
            raise RuntimeError("No patches found for split=" + split + " in " +
                                PATCH_DIR_OPT_SAR + ". Run Stage 5b first.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        feats = d["features"].astype(np.float32)  # [15, H, W]
        mean = feats.mean(axis=(1, 2), keepdims=True)
        std = feats.std(axis=(1, 2), keepdims=True) + 1e-6
        feats = (feats - mean) / std

        pixel_mask = d["pixel_mask"].astype(np.float32)
        patch_label = int(d["label"])
        return torch.from_numpy(feats), torch.from_numpy(pixel_mask), patch_label


def build_unet_opt_sar_model():
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,  # ImageNet weights don't apply to a 15-channel input
        in_channels=OPT_SAR_IN_CHANNELS,
        classes=1,
    )
    return model.to(DEVICE)


def train_epoch_opt_sar(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for feats, pixel_mask, _ in loader:
        feats, pixel_mask = feats.to(DEVICE), pixel_mask.to(DEVICE)
        optimizer.zero_grad()
        logits = model(feats).squeeze(1)
        loss = criterion(logits, pixel_mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * feats.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_opt_sar(model, loader):
    model.eval()
    all_patch_preds, all_patch_labels = [], []
    for feats, pixel_mask, patch_label in loader:
        feats = feats.to(DEVICE)
        logits = model(feats).squeeze(1)
        pixel_probs = torch.sigmoid(logits)
        pixel_preds = (pixel_probs > 0.5).float().cpu()
        patch_pred = (pixel_preds.mean(dim=(1, 2)) > 0.5).long().numpy()
        all_patch_preds.extend(patch_pred.tolist())
        all_patch_labels.extend(patch_label.numpy().tolist())
    return np.array(all_patch_labels), np.array(all_patch_preds)


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    train_ds = PatchDatasetOptSar("train")
    val_ds = PatchDatasetOptSar("val")
    test_ds = PatchDatasetOptSar("test")

    train_loader = DataLoader(train_ds, batch_size=OPT_SAR_BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=OPT_SAR_BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=OPT_SAR_BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_unet_opt_sar_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=OPT_SAR_LR)
    criterion = nn.BCEWithLogitsLoss()

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="unet_optical_sar"):
        mlflow.log_params(dict(
            encoder="resnet18", in_channels=OPT_SAR_IN_CHANNELS, epochs=OPT_SAR_EPOCHS,
            batch_size=OPT_SAR_BATCH_SIZE, lr=OPT_SAR_LR, device=str(DEVICE),
        ))

        for epoch in range(1, OPT_SAR_EPOCHS + 1):
            train_loss = train_epoch_opt_sar(model, train_loader, optimizer, criterion)
            val_labels, val_preds = evaluate_opt_sar(model, val_loader)
            val_acc = accuracy_score(val_labels, val_preds) if len(val_labels) else float("nan")
            print("Epoch " + str(epoch) + "/" + str(OPT_SAR_EPOCHS) +
                  "  train_loss=" + "{:.4f}".format(train_loss) +
                  "  val_acc=" + "{:.4f}".format(val_acc))
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        test_labels, test_preds = evaluate_opt_sar(model, test_loader)
        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_per_class = f1_score(test_labels, test_preds, average=None, labels=[0, 1])
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric("test_f1_" + cname, f1c)

        print("\n=== UNet, optical+SAR (15ch) -- Test Set ===")
        print("Accuracy: {:.4f}".format(acc))
        print("Macro F1: {:.4f}".format(f1_macro))
        print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, labels=[0, 1]))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "unet_optical_sar",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        })

    return model


if __name__ == "__main__":
    unet_opt_sar_model = main()


Using device: cuda
Epoch 1/15  train_loss=0.5020  val_acc=0.8471
Epoch 2/15  train_loss=0.4384  val_acc=0.8709
Epoch 3/15  train_loss=0.4196  val_acc=0.8709
Epoch 4/15  train_loss=0.4067  val_acc=0.8442
Epoch 5/15  train_loss=0.3952  val_acc=0.8423
Epoch 6/15  train_loss=0.3885  val_acc=0.8418
Epoch 7/15  train_loss=0.3801  val_acc=0.9028
Epoch 8/15  train_loss=0.3736  val_acc=0.8804
Epoch 9/15  train_loss=0.3673  val_acc=0.8857
Epoch 10/15  train_loss=0.3583  val_acc=0.8938
Epoch 11/15  train_loss=0.3552  val_acc=0.8595
Epoch 12/15  train_loss=0.3500  val_acc=0.8742
Epoch 13/15  train_loss=0.3421  val_acc=0.8923
Epoch 14/15  train_loss=0.3375  val_acc=0.9033
Epoch 15/15  train_loss=0.3250  val_acc=0.9028

=== UNet, optical+SAR (15ch) -- Test Set ===
Accuracy: 0.9027
Macro F1: 0.9024
              precision    recall  f1-score   support

non_cropland       0.94      0.86      0.90      1179
    cropland       0.87      0.95      0.91      1227

    accuracy                           0.

In [ ]:
#prithvi
!pip install -q -U transformers huggingface_hub



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 13.4 MB/s eta 0:00:00


In [ ]:
# Model -5  REAL PRITHVI-100M (replacing the SSL4EO-S12 substitute)

from huggingface_hub import hf_hub_download
import importlib.util

ckpt_path = hf_hub_download("ibm-nasa-geospatial/Prithvi-EO-1.0-100M", "Prithvi_100M.pt")
code_path = hf_hub_download("ibm-nasa-geospatial/Prithvi-EO-1.0-100M", "prithvi_mae.py")

# Load their model class directly from the downloaded file, bypassing
# AutoModel/trust_remote_code entirely
spec = importlib.util.spec_from_file_location("prithvi_mae", code_path)
prithvi_mae = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prithvi_mae)

print([n for n in dir(prithvi_mae) if not n.startswith("_")])

Prithvi_100M.pt: reconstructing file:   0%|          |  0.00B /  454MB            

Prithvi_100M.pt: downloading bytes:           |  0.00B            

prithvi_mae.py:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

['Block', 'List', 'LocationEncoder', 'MAEDecoder', 'PatchEmbed', 'PrithviMAE', 'PrithviViT', 'TemporalEncoder', 'Tuple', 'get_1d_sincos_pos_embed_from_grid', 'get_3d_sincos_pos_embed', 'logging', 'nn', 'np', 'partial', 'rearrange', 'to_2tuple', 'torch']


In [ ]:
from huggingface_hub import list_repo_files

repo_id = "ibm-nasa-geospatial/Prithvi-EO-1.0-100M"

files = list_repo_files(repo_id)
for f in files:
    print(f)

.gitattributes
GFM.png
Prithvi_100M.pt
Prithvi_EO_V1_100M.pt
README.md
config.json
config.yaml
examples/HLS.L30.T13REN.2018013T172747.v2.0.B02.B03.B04.B05.B06.B07_cropped.tif
examples/HLS.L30.T13REN.2018029T172738.v2.0.B02.B03.B04.B05.B06.B07_cropped.tif
examples/HLS.L30.T13REN.2018061T172724.v2.0.B02.B03.B04.B05.B06.B07_cropped.tif
inference.py
prithvi_mae.py
requirements.txt


In [ ]:
import yaml, torch
cfg_path = hf_hub_download(repo_id, "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

print(cfg)


config.yaml:   0%|          | 0.00/653 [00:00<?, ?B/s]

{'model_args': {'decoder_depth': 8, 'decoder_embed_dim': 512, 'decoder_num_heads': 16, 'depth': 12, 'embed_dim': 768, 'img_size': 224, 'in_chans': 6, 'num_frames': 3, 'num_heads': 12, 'patch_size': 16, 'tubelet_size': 1}, 'train_params': {'bands': ['B02', 'B03', 'B04', 'B05', 'B06', 'B07'], 'data_mean': [775.2290211032589, 1080.992780391705, 1228.5855250417867, 2497.2022620507532, 2204.2139147975554, 1610.8324823273745], 'data_std': [1281.526139861424, 1270.0297974547493, 1399.4802505642526, 1368.3446143747644, 1291.6764008585435, 1154.505683480695], 'mask_ratio': 0.75, 'random_cropping': True}}


In [ ]:
import inspect

sig = inspect.signature(prithvi_mae.PrithviViT.__init__)
valid_args = set(sig.parameters.keys())
model_args = {k: v for k, v in cfg['model_args'].items() if k in valid_args}
print("Using args:", model_args)

prithvi_encoder = prithvi_mae.PrithviViT(**model_args)

Using args: {'depth': 12, 'embed_dim': 768, 'img_size': 224, 'in_chans': 6, 'num_frames': 3, 'num_heads': 12, 'patch_size': 16}


In [ ]:
state_dict = torch.load(ckpt_path, map_location="cpu")
# strip decoder/mask-token keys — we only want encoder weights
encoder_sd = {k: v for k, v in state_dict.items() if not k.startswith("decoder")}
missing, unexpected = prithvi_encoder.load_state_dict(encoder_sd, strict=False)
print("missing:", missing[:5], "... unexpected:", unexpected[:5])

missing: [] 
unexpected: []


In [ ]:
# strip the "encoder." prefix so keys match your bare PrithviViT instance
encoder_sd = {
    k[len("encoder."):]: v
    for k, v in state_dict.items()
    if k.startswith("encoder.")
}
missing, unexpected = prithvi_encoder.load_state_dict(encoder_sd, strict=False)
print("missing:", missing, "\nunexpected:", unexpected)

missing: [] 
unexpected: []


In [ ]:
import torch.nn.functional as F
def prithvi_prep_batch(feats):
    feats = F.interpolate(feats, size=(PRITHVI_IMG_SIZE, PRITHVI_IMG_SIZE),
                           mode="bilinear", align_corners=False)
    mean = feats.mean(dim=(2, 3), keepdim=True)
    std = feats.std(dim=(2, 3), keepdim=True) + 1e-6
    feats = (feats - mean) / std
    return feats.unsqueeze(2).repeat(1, 1, 3, 1, 1)  # [B, 6, 3, 224, 224] -- matches num_frames=3


class PrithviClassifier(nn.Module):
    def __init__(self, encoder, embed_dim=768, num_classes=2):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x arrives already prepped by prithvi_prep_batch: [B, 6, 3, 224, 224]
        out = self.encoder(x)
        if hasattr(out, "last_hidden_state"):
            feats = out.last_hidden_state
        elif isinstance(out, (tuple, list)):
            feats = out[0]
        else:
            feats = out
        pooled = feats.mean(dim=1) if feats.dim() == 3 else feats
        return self.head(pooled)


prithvi_model = PrithviClassifier(prithvi_encoder).to(DEVICE)

In [ ]:
#Load the REAL Prithvi-EO-1.0-100M + band remap

# Model: ibm-nasa-geospatial/Prithvi-EO-1.0-100M. this is the weight
# The HF repo was renamed since the original Aug-2023 IBM/NASA release but the checkpoint is the same.
# It's a TEMPORAL ViT: input is shaped (B, C=6, T, H=224, W=224). It was
# pretrained with T=3 timesteps, but the model card explicitly supports
# T=1 for static imagery, which is what's used here, my patches are
# single-season, not a stacked multi-temporal tensor. Stacking the 4
# seasons into one T=4 input per location

# If AutoModel/trust_remote_code fails
# outright, the fallback is to hf_hub_download the raw Prithvi_100M.pt +
# prithvi_mae.py files from the same repo and instantiate the model
# class directly from that file
# IBM/NASA ship, so there's no wrapper-API guessing involved.

from transformers import AutoModel

PRITHVI_MODEL_ID = "ibm-nasa-geospatial/Prithvi-EO-1.0-100M"
PRITHVI_EMBED_DIM = 768     # from the repo's config.yaml (model_args.embed_dim)
PRITHVI_IMG_SIZE = 224

# Official HLS band order Prithvi expects: B02, B03, B04, B05, B06, B07 =
# Blue, Green, Red, NARROW NIR, SWIR1, SWIR2. In HLS's Sentinel-2 band
# naming, "narrow NIR" (B05) is Sentinel-2's B8A -- NOT the broad NIR
# band B8 used earlier. Getting this wrong (using B8 instead
# of B8A) is a common mistake,it won't error, it'll just
# quietly feed the model out-of-distribution input.
# BLUE, GREEN, RED, RE1, RE2, RE3, NIR, RE4, SWIR1, SWIR2 = range(10)  (Stage 3, reused)
PRITHVI_BAND_IDX = [BLUE, GREEN, RED, RE4, SWIR1, SWIR2]  # RE4 is Stage 3's name for B8A


def remap_to_prithvi_bands(features_13ch):
    """features_13ch: [13, H, W] (10 raw S2 bands + NDVI/NDWI/EVI).
    Returns [6, H, W], raw scale (NOT divided by
    10000), in Prithvi's expected band order."""
    return features_13ch[PRITHVI_BAND_IDX]


try:
    prithvi_backbone = AutoModel.from_pretrained(PRITHVI_MODEL_ID, trust_remote_code=True)
    print("Loaded real Prithvi-EO-1.0-100M weights.")
except Exception as e:
    raise RuntimeError(
        "Could not load " + PRITHVI_MODEL_ID + " via AutoModel(trust_remote_code=True): " +
        str(e) + "\nFallback: hf_hub_download('" + PRITHVI_MODEL_ID + "', 'Prithvi_100M.pt') "
        "and 'prithvi_mae.py' directly, import the class from the downloaded file, and "
        "load the state dict manually."
    )


class PrithviClassifier(nn.Module):
    """Linear-probe / fine-tune head on top of the Prithvi encoder."""
    def __init__(self, backbone, embed_dim=PRITHVI_EMBED_DIM, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x: [B, 6, T, 224, 224]
        out = self.backbone(x)
        if hasattr(out, "last_hidden_state"):
            feats = out.last_hidden_state
        elif isinstance(out, (tuple, list)):
            feats = out[0]
        elif torch.is_tensor(out):
            feats = out
        else:
            raise RuntimeError(
                "Unrecognized Prithvi backbone output type: " + str(type(out)) +
                ". Inspect it directly (e.g. out.keys() if dict-like, or dir(out)) "
                "and adjust this branch"
            )
        pooled = feats.mean(dim=1) if feats.dim() == 3 else feats
        return self.head(pooled)


prithvi_model = PrithviClassifier(prithvi_backbone).to(DEVICE)
n_params = sum(p.numel() for p in prithvi_model.parameters())
print("Prithvi classifier built: {:.1f}M params total.".format(n_params / 1e6))

# --- Fallback path if the cell above fails (commented out) ---
# from huggingface_hub import hf_hub_download
# ckpt_path = hf_hub_download(PRITHVI_MODEL_ID, "Prithvi_100M.pt")
# code_path = hf_hub_download(PRITHVI_MODEL_ID, "prithvi_mae.py")
# cfg_path  = hf_hub_download(PRITHVI_MODEL_ID, "Prithvi_100M_config.yaml")
# # -> import prithvi_mae.py as a module (importlib.util.spec_from_file_location),
# #    instantiate its ViT/MAE encoder class with the model_args from the
# #    config above, then encoder.load_state_dict(torch.load(ckpt_path,
# #    map_location="cpu"), strict=False). Bypasses trust_remote_code
# #    entirely, using the exact class IBM/NASA ship, at the cost of a
# #    few more lines to strip decoder/mask-token keys you don't need.


config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

RuntimeError: Could not load ibm-nasa-geospatial/Prithvi-EO-1.0-100M via AutoModel(trust_remote_code=True): 'NoneType' object cannot be interpreted as an integer
Fallback: hf_hub_download('ibm-nasa-geospatial/Prithvi-EO-1.0-100M', 'Prithvi_100M.pt') and 'prithvi_mae.py' directly, import the class from the downloaded file, and load the state dict manually -- see the comment block above this cell.

In [ ]:
# Fine-tune real Prithvi: 2-epoch linear-probe warmup, then
# unfreeze at lr=1e-4. Same strategy, same split, same metrics as your
# existing VIT(SSL4EO-S12 substitute) run, so the two results are
# directly comparable.

PATCH_DIR = "/content/patches"          # ORIGINAL optical-only patches (Prithvi doesn't use SAR)
RESULTS_PATH = "/content/results.json"
CLASS_NAMES = ["non_cropland", "cropland"]

PRITHVI_BATCH_SIZE = 16       # drop to 8 if you hit CUDA OOM on the free T4 (16GB) --
                              # Prithvi-100M's ViT-Base encoder has ~4-5x the params
                              # of the ViT-Small substitute you already benchmarked
PRITHVI_EPOCHS = 6            # 2 frozen + 4 unfrozen; deliberately lighter than
                              # Stage 4c's 10 epochs; see the time-budget note below
PRITHVI_LR_HEAD = 1e-3        # linear-probe phase (head only)
PRITHVI_LR_FINETUNE = 1e-4    # requested fine-tune LR, once unfrozen
PRITHVI_FREEZE_EPOCHS = 2
USE_AMP = True                # mixed precision; meaningfully faster on a T4, low risk


class PrithviPatchDataset(Dataset):
    def __init__(self, split):
        self.paths = sorted(glob.glob(os.path.join(PATCH_DIR, split, "*.npz")))
        if len(self.paths) == 0:
            raise RuntimeError("No patches found for split=" + split + " in " +
                                PATCH_DIR + ". Run Stage 3 first.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        feats13 = d["features"].astype(np.float32)     # [13, 64, 64]
        feats6 = remap_to_prithvi_bands(feats13)        # [6, 64, 64], reused mapping
        label = int(d["label"])
        return torch.from_numpy(feats6), label


def prithvi_prep_batch(feats):
    """[B, 6, 64, 64] -> [B, 6, 1, 224, 224], per-sample normalized
    (per-patch mean/std -- the same simplification your notebook
    already uses elsewhere, NOT Prithvi's published per-band HLS
    statistics; swap in the exact data_mean/data_std from the repo's
    config.yaml if you want closer fidelity to the pretraining
    distribution)."""
    feats = F.interpolate(feats, size=(PRITHVI_IMG_SIZE, PRITHVI_IMG_SIZE),
                           mode="bilinear", align_corners=False)
    mean = feats.mean(dim=(2, 3), keepdim=True)
    std = feats.std(dim=(2, 3), keepdim=True) + 1e-6
    feats = (feats - mean) / std
    return feats.unsqueeze(2)  # insert T=1 dim -> [B, 6, 1, 224, 224]


def set_prithvi_encoder_trainable(model, trainable):
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            param.requires_grad = trainable


def train_epoch_prithvi(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    for feats, labels in loader:
        feats = prithvi_prep_batch(feats).to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(feats)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * feats.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_prithvi(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for feats, labels in loader:
        feats = prithvi_prep_batch(feats).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(feats)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_preds)


def append_results(entry):
    results = []
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            results = json.load(f)
    results.append(entry)
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)


def main():
    train_ds = PrithviPatchDataset("train")
    val_ds = PrithviPatchDataset("val")
    test_ds = PrithviPatchDataset("test")

    train_loader = DataLoader(train_ds, batch_size=PRITHVI_BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=PRITHVI_BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=PRITHVI_BATCH_SIZE, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    mlflow.set_experiment("crop_classification")
    with mlflow.start_run(run_name="foundation_model_prithvi_100m_real"):
        mlflow.log_params(dict(
            base_model=PRITHVI_MODEL_ID, epochs=PRITHVI_EPOCHS,
            batch_size=PRITHVI_BATCH_SIZE, lr_finetune=PRITHVI_LR_FINETUNE,
            freeze_encoder_epochs=PRITHVI_FREEZE_EPOCHS, device=str(DEVICE),
        ))

        for epoch in range(1, PRITHVI_EPOCHS + 1):
            frozen = epoch <= PRITHVI_FREEZE_EPOCHS
            set_prithvi_encoder_trainable(prithvi_model, trainable=not frozen)
            lr = PRITHVI_LR_HEAD if frozen else PRITHVI_LR_FINETUNE
            optimizer = torch.optim.AdamW(
                [p for p in prithvi_model.parameters() if p.requires_grad], lr=lr
            )
            train_loss = train_epoch_prithvi(prithvi_model, train_loader, optimizer, criterion, scaler)
            val_labels, val_preds = evaluate_prithvi(prithvi_model, val_loader)
            val_acc = accuracy_score(val_labels, val_preds) if len(val_labels) else float("nan")
            phase = "linear-probe" if frozen else "fine-tune"
            print("Epoch " + str(epoch) + "/" + str(PRITHVI_EPOCHS) + " [" + phase + "]" +
                  "  train_loss=" + "{:.4f}".format(train_loss) +
                  "  val_acc=" + "{:.4f}".format(val_acc))
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        test_labels, test_preds = evaluate_prithvi(prithvi_model, test_loader)
        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_per_class = f1_score(test_labels, test_preds, average=None, labels=[0, 1])
        cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1_macro", f1_macro)
        for cname, f1c in zip(CLASS_NAMES, f1_per_class):
            mlflow.log_metric("test_f1_" + cname, f1c)

        print("\n=== Real Prithvi-EO-1.0-100M -- Test Set ===")
        print("Accuracy: {:.4f}".format(acc))
        print("Macro F1: {:.4f}".format(f1_macro))
        print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, labels=[0, 1]))
        print("Confusion matrix (rows=true, cols=pred):")
        print(cm)

        append_results({
            "model": "foundation_model_prithvi_100m_real",
            "test_accuracy": float(acc),
            "test_f1_macro": float(f1_macro),
            "test_f1_per_class": {c: float(v) for c, v in zip(CLASS_NAMES, f1_per_class)},
            "confusion_matrix": cm.tolist(),
            "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        })

    return prithvi_model


if __name__ == "__main__":
    prithvi_model = main()


/tmp/ipykernel_940/803648177.py:120: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


Epoch 1/6 [linear-probe]  train_loss=0.6525  val_acc=0.6770
Epoch 2/6 [linear-probe]  train_loss=0.5362  val_acc=0.7394
Epoch 3/6 [fine-tune]  train_loss=0.4670  val_acc=0.7575
Epoch 4/6 [fine-tune]  train_loss=0.4511  val_acc=0.7594
Epoch 5/6 [fine-tune]  train_loss=0.4407  val_acc=0.7694
Epoch 6/6 [fine-tune]  train_loss=0.4318  val_acc=0.7818

=== Real Prithvi-EO-1.0-100M -- Test Set ===
Accuracy: 0.8354
Macro F1: 0.8349
              precision    recall  f1-score   support

non_cropland       0.86      0.80      0.83      1179
    cropland       0.82      0.87      0.84      1227

    accuracy                           0.84      2406
   macro avg       0.84      0.83      0.83      2406
weighted avg       0.84      0.84      0.84      2406

Confusion matrix (rows=true, cols=pred):
[[ 941  238]
 [ 158 1069]]
